# NanoForge on Kaggle TPU x8

In Kaggle, import this notebook from GitHub or upload it, enable **Internet**, and choose the **TPU x8** accelerator in Notebook settings. Run cells in order. TPU hardware availability and quotas depend on your Kaggle account.

This uses a separate PyTorch/XLA process, all eight replicas, **BF16** autocast and uint16 token caches. CUDA FP16 settings do not apply to TPUs. Model parameters and optimizer state remain FP32, and exported `.pt` files work in the local Studio.

Eight cores hold eight complete replicas, not one pooled-memory model. Start with Mini. Custom architectures must have fewer than one billion parameters and fit **each core's** memory. First-step XLA compilation can take several minutes. This notebook includes a hardware smoke test because TPU kernels cannot be verified on the CPU-only development machine.

`/kaggle/working` contains outputs; preserve/download checkpoints or save a notebook version with outputs before ending your session. Internet is needed for cloning, dependency installation and dataset imports.

In [ ]:
from pathlib import Path
import importlib.metadata, json, os, subprocess, sys
REPO = Path('/kaggle/working/nanoforge')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/bananaosint/nanoforge.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)
os.environ['PJRT_DEVICE'] = 'TPU'
os.environ.pop('XLA_USE_BF16',None)
os.environ.pop('XLA_DOWNCAST_BF16',None)
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
os.environ['PIP_NO_CACHE_DIR'] = '1'
temp = Path('/kaggle/working/nanoforge-install-tmp'); temp.mkdir(exist_ok=True)
os.environ['TMPDIR'] = str(temp)

## Install a matching PyTorch / PyTorch-XLA pair

Reuse Kaggle's installed pair when major/minor versions match and are at least 2.8. Otherwise install the documented 2.8.0 CPU PyTorch + TPU XLA pair. CUDA wheels are unnecessary. Installation runs in subprocesses; do not initialize TensorFlow's TPU runtime or call `torch_xla.device()` in this notebook process. Training will spawn its own fresh workers.

If you already initialized a TPU runtime in other cells, restart the Kaggle session and start here. An ABI/undefined-symbol error indicates mismatched packages; use the explicit 2.8.0 installation from `docs/kaggle-tpu.md`.

In [ ]:
def installed(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

def minor(version):
    return tuple(int(part) for part in version.split('.')[:2]) if version else None

torch_version, xla_version = installed('torch'), installed('torch_xla')
matching = torch_version and xla_version and minor(torch_version)==minor(xla_version) and (2,8)<=minor(xla_version)<(3,0)
if not matching:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--upgrade','torch==2.8.0',
                    '--index-url','https://download.pytorch.org/whl/cpu'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--upgrade','torch_xla[tpu]==2.8.0',
                    '-f','https://storage.googleapis.com/libtpu-releases/index.html'],check=True)
if matching and installed('libtpu') is None:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir',f'torch_xla[tpu]=={xla_version}',
                    '-f','https://storage.googleapis.com/libtpu-releases/index.html'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','-r','backend/requirements.txt'],check=True)
print('torch:',installed('torch'),'torch_xla:',installed('torch_xla'))

## Verify eight-core TPU training and Studio export

This two-step test really launches every TPU replica, checks that exactly eight are present, trains using BF16, exports CPU tensors and loads the result for CPU generation in the Studio backend. A failure stops the notebook before a longer run. It is expected to take longer than a CPU smoke test because compilation dominates tiny runs.

In [ ]:
test_env = {**os.environ,'NANOFORGE_TEST_TPU':'1','PJRT_DEVICE':'TPU'}
subprocess.run([sys.executable,'-m','pytest','backend/tests/test_tpu_hardware.py','-q','-s'],check=True,env=test_env)

## Import and pre-tokenize

Dolly is a manageable example. The full Cosmopedia/Wikipedia imports are too large for ordinary notebook disk quotas. Imported prompt/answer pairs can be used for pretraining or completion-masked SFT. The pretokenization command prints `uint16`; only sampled batches expand to int64.

In [ ]:
RUN_HOME = Path('/kaggle/working/nanoforge-run')
RUN_HOME.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','backend.importers','--dataset','dolly','--home',str(RUN_HOME)],check=True)
subprocess.run([sys.executable,'-m','backend.pretokenize','--home',str(RUN_HOME),'--tokenizer','magic8k','--dataset','dolly.jsonl'],check=True)

## Train with BF16 on all eight cores

The default profile uses Mini, context 256, **batch 4 per core**, and accumulation 2: effective global batch **64** on eight replicas. Gradients are averaged across replicas before clipping and updating. Change the dimensions for a custom model; the backend checks the exact count before allocating weights. Leave precision `bfloat16` for TPU.

Change `mode` to `sft` for imported prompt/answer pairs, or `dpo` with a preference dataset. DPO also holds a frozen reference model, so it needs extra memory. For resume, set `RESUME` to a complete NanoForge `.pt` checkpoint and retain its tokenizer. Architecture comes from the checkpoint; `max_steps` means additional steps.

By default, only the latest completed checkpoint from this invocation is retained (`--keep-last 1`). Older runs are preserved. Checkpoints need disk space for both the previous file and the new temporary file during atomic saving. Interrupting a distributed job may terminate workers immediately; resume the last **completed** checkpoint, not an unfinished `.pending` file.

In [ ]:
config = json.loads((REPO/'configs/kaggle-tpu.json').read_text())
# Optional custom architecture (must fit each TPU core, not total pooled memory):
# config.update(preset='Custom', model=dict(n_layer=12,n_head=8,n_embd=768,block_size=256,dropout=0.1))
# config['batch_size'] = 1
# config['gradient_accumulation'] = 8
# config['max_steps'] = 100
# config['mode'] = 'sft'
config_file = RUN_HOME/'training-config.json'
config_file.write_text(json.dumps(config,indent=2))
RESUME = None  # Example: Path('/kaggle/input/my-checkpoint/pretrain-...-s1000.pt')
command = [sys.executable,'-m','backend.train_xla','--config',str(config_file),
           '--home',str(RUN_HOME),'--expected-replicas','8','--keep-last','1']
if RESUME is not None: command += ['--resume',str(RESUME)]
subprocess.run(command,check=True,env={**os.environ,'PJRT_DEVICE':'TPU'})

## Download and open in Studio

Download the `.pt` using the link or Kaggle's Output file panel, then choose **Train & Monitor → Import .pt checkpoint** in your updated local Studio. It reconstructs the JSON metadata and switches inference back to local precision. Select local dataset files before continuing training.

Alternatively copy both `.pt` and `.json` to the Studio's `checkpoints/` folder while it is stopped. Tokenizer, architecture, weights and optimizer state travel with the `.pt`; datasets and the Kaggle metrics log do not. No ZIP is made by default, avoiding a second multi-gigabyte copy on Kaggle's disk.

In [ ]:
import torch
from IPython.display import FileLink, display
manifest = json.loads((RUN_HOME/'latest-checkpoint.json').read_text())
checkpoint = Path(manifest['checkpoint'])
payload = torch.load(checkpoint,map_location='cpu',weights_only=True,mmap=True)
assert {'model_config','tokenizer_state','model_state_dict','optimizer_state_dict','step','val_loss','version_tag'} <= payload.keys()
assert payload['world_size']==8
assert all(t.device.type=='cpu' and t.dtype==torch.float32 for t in payload['model_state_dict'].values())
print('Portable checkpoint:',checkpoint.name,'step:',payload['step'])
del payload
display(FileLink(str(checkpoint)))
display(FileLink(str(Path(manifest['metadata']))))